# Практическое занятие №1: Многопоточность (Threads) в Python

## 1. Введение: Блокирующие операции и параллелизм
Код, который мы пишем, по умолчанию выполняется строго последовательно. Однако в реальной разработке программы часто сталкиваются с **блокирующими операциями** — задачами, при выполнении которых программа вынуждена простаивать. Это может быть ожидание ответа от базы данных, скачивание файла по сети или чтение тяжелого файла с жесткого диска.

Если в этот момент программа не может выполнять другой код, её эффективность падает. Для решения этой проблемы в операционных системах существуют механизмы распараллеливания задач: **процессы** и **потоки**.

## 2. Процессы и Потоки: В чем разница?

* **Процесс (Process)** — это экземпляр запущенной программы. Операционная система выделяет каждому процессу **собственное изолированное адресное пространство** в оперативной памяти. Процессы полностью независимы: если один процесс завершится с ошибкой, остальные продолжат работу. Однако из-за этой изоляции обмен данными между процессами требует значительных системных ресурсов.
* **Поток (Thread)** — это нить выполнения **внутри** одного процесса. Внутри одного процесса может работать множество потоков. Главное отличие потоков заключается в том, что они **разделяют общую память** процесса.
  * *Плюсы:* Потоки создаются очень быстро и потребляют минимум ресурсов (их называют "легковесными"). Они могут мгновенно обмениваться данными через общие переменные.
  * *Минусы:* Так как память общая, разработчику нужно следить за тем, чтобы два потока не попытались изменить одну и ту же переменную одновременно (состояние гонки / race condition).

## 3. Где правильно использовать потоки?
Потоки идеально подходят для задач ввода-вывода (**I/O-bound** задач), где процессор большую часть времени ожидает данных.

* **Разработка графических интерфейсов (GUI):** В любом приложении с интерфейсом существует понятие "Главный поток" (Main Thread). Его единственная задача — отрисовывать окна и реагировать на клики мыши (Event Loop). Если вы начнете скачивать тяжелый файл в главном потоке, он заблокируется, и интерфейс программы "зависнет" (появится надпись "Не отвечает"). Правильный архитектурный подход — всегда выносить любые тяжелые вычисления и сетевые запросы в отдельные фоновые потоки (Worker Threads).
* **Сетевые запросы и парсинг:** Если вам нужно сделать 100 запросов к API, последовательное выполнение займет много времени, так как процессор будет просто ждать ответа сервера на каждый запрос. Использование потоков позволит отправить множество запросов параллельно.

## 4. Особенность Python: GIL (Global Interpreter Lock)
Может показаться, что потоки — идеальное решение для ускорения любых программ. Но в стандартной реализации Python (CPython) есть архитектурная особенность — **GIL (Глобальная блокировка интерпретатора)**.

Python автоматически управляет памятью с помощью подсчета ссылок. Чтобы избежать ошибок при одновременном обращении нескольких потоков к одной переменной, создатели Python ввели жесткое правило: **в любой момент времени выполнять байт-код Python может только один поток**. GIL — это "мьютекс" (замок), который поток должен захватить перед тем, как выполнить команду.

**Как это влияет на наш код?**
1. **I/O-bound задачи (Сеть/Файлы):** Потоки работают отлично. Когда поток отправляет запрос в сеть и начинает ждать ответа, он *отпускает* GIL. В это время другой поток берет GIL и выполняет свою работу.
2. **CPU-bound задачи (Математика/Алгоритмы):** Если вы попытаетесь ускорить с помощью потоков сложные вычисления (где процессор загружен на 100% и не простаивает), потоки начнут постоянно бороться за GIL. Операционная система будет тратить время на переключение контекста между потоками, и в результате **многопоточный код отработает медленнее, чем однопоточный**.


# ОДНОПОТОЧНОЕ ВЫПОЛНЕНИЕ ПРОГРАММЫ

In [ ]:
memory_process = [1,2,3,4,5,6]

In [ ]:
memory_process

In [ ]:
def io_bound_task(task_id, delay):
    print(f"Основной поток програмы {task_id}: начал ожидание (отправил запрос)...")
    time.sleep(delay)  # В этот момент поток отпускает GIL!
    memory_process.append(i)
    memory_process[i] = memory_process[i]**2
    print(f"Основной поток програмы {task_id}: завершил работу (получил ответ)!")

In [ ]:
import time
import threading

memory_process = [1,2,3,4,5,6] # общая память процесса

# Функция, которая имитирует задачу ввода-вывода (I/O-bound)
# Например: сетевой запрос, который заставляет программу ждать
def io_bound_task(task_id, delay):
    print(f"Основной поток програмы {task_id}: начал ожидание (отправил запрос)...")
    time.sleep(delay)  # В этот момент поток отпускает GIL!
    memory_process.append(i)
    memory_process[i] = memory_process[i]**2
    print(memory_process)
    print(f"Основной поток програмы {task_id}: завершил работу (получил ответ)!")

print("=== 1. ПОСЛЕДОВАТЕЛЬНОЕ ВЫПОЛНЕНИЕ (Один поток) ===")
start_seq = time.time()
for i in range(3):
    io_bound_task(task_id=i, delay=2)
print(f"Итоговое время: {time.time() - start_seq:.2f} сек.\n")
print(memory_process)

# МНОГОПОТОЧНОЕ ВЫПОЛНЕНИЕ

In [ ]:
print("=== 2. МНОГОПОТОЧНОЕ ВЫПОЛНЕНИЕ (Модуль threading) ===")
start_thread = time.time()

# Список для хранения объектов потоков
threads = []
memory_process = [1,2,3,4,5,6] # общая память процесса

# Шаг 1: Создание и запуск потоков
for i in range(3):
    # Создаем объект потока.
    # target - какую функцию выполнить, args - кортеж аргументов для функции
    t = threading.Thread(target=io_bound_task, args=(i, 2))
    threads.append(t)
    # Метод start() сообщает ОС, что поток готов к выполнению
    t.start()
    t.join()

print(f"Итоговое время: {time.time() - start_thread:.2f} сек.\n")
print("Вывод: Для задач ожидания потоки дают кратное ускорение!")
print(memory_process)

In [ ]:
print("=== 2. МНОГОПОТОЧНОЕ ВЫПОЛНЕНИЕ (Модуль threading) ===")
start_thread = time.time()

# Список для хранения объектов потоков
threads = []

# Шаг 1: Создание и запуск потоков
for i in range(3):
    # Создаем объект потока.
    # target - какую функцию выполнить, args - кортеж аргументов для функции
    t = threading.Thread(target=io_bound_task, args=(i, 2))
    threads.append(t)
    # Метод start() сообщает ОС, что поток готов к выполнению
    t.start()

# Шаг 2: Ожидание завершения потоков
# Если не вызвать join(), главная программа пойдет дальше, не дожидаясь фоновых задач
for t in threads:
    t.join()

print(f"Итоговое время: {time.time() - start_thread:.2f} сек.\n")
print("Вывод: Для задач ожидания потоки дают кратное ускорение!")
print(memory_process)

# Пример работы с MLPRegressor

In [ ]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split

In [ ]:
print("Генерация данных (это займет пару секунд)...")
X, y = make_regression(n_samples=1000, n_features=30, noise=0.5, random_state=42)
print("Данные готовы!\n")


In [ ]:
print("Создаем нейросеть: 2 слоя (50 и 20 нейронов), функция активации 'tanh'")
model = MLPRegressor(
    hidden_layer_sizes=(50, 20),
    activation='tanh',
    max_iter=300, # Максимальное число эпох обучения
    random_state=42
)


In [ ]:
# Обучаем модель (метод fit)
model.fit(X, y)
print("Модель успешно обучена!")

## 6. Практическое задание

Вам предстоит обучить 9 нейронных сетей с разными гиперпараметрами на "тяжелом" наборе данных, а затем визуализировать качество их обучения.

**Ваши задачи:**
1. Сформировать конфигурации для **9 различных нейросетей**:
   * Сети 1-3: Одинаковое количество слоев (один), но разное число нейронов (например, 100, 200, 300).
   * Сети 4-6: Фиксированное число нейронов (например, 150), но разные функции активации (`relu`, `tanh`, `logistic`).
   * Сети 7-9: Разное количество скрытых слоев (от 1 до 3 слоев).
2. Написать код для **последовательного** обучения сетей и замерить время.
3. Написать код для **многопоточного** обучения (модуль `threading`) и замерить время.
4. **Визуализация:** Построить графики "Предсказание от Истины" (True vs Predicted) для каждой из 9 обученных моделей с помощью `matplotlib`. На таком графике по оси X откладываются истинные значения, а по оси Y — предсказания сети. Чем ближе точки к диагонали, тем точнее сеть.

In [ ]:
#  импорты библиотек
import time
import threading
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split

In [ ]:
# 1. Генерация тяжелого датасета и разбиение на train/test
print("Генерация данных (это займет пару секунд)...")
X, y = make_regression(n_samples=5000, n_features=30, noise=0.5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Данные готовы!\n")


In [ ]:
# Формирование 9 конфигураций (в виде простых списков: [слои, активация])
networks_settings = [
    # Группа 1: Разное число нейронов
    [(100,), 'relu'],
    [(200,), 'relu'],
    [(300,), 'relu'],

    # Группа 2: Разные функции активации
    [(150,), 'relu'],
    [(150,), 'tanh'],
    [(150,), 'logistic'],

    # Группа 3: Разное число слоев
    [(50,), 'relu'],
    [(50, 50), 'relu'],
    [(50, 50, 50), 'relu']
]

In [ ]:
# Создаем пустой список из 9 элементов для хранения результатов
predictions_results = [None] * 9

In [ ]:
# Функция для обучения. Сохраняет предсказания прямо в словарь конфигурации.
# Функция, которая обучает одну модель и сохраняет результат
def train_and_predict(index, layers, activation):
    # Создаем модель с заданными параметрами
    model = MLPRegressor(hidden_layer_sizes=layers, activation=activation, max_iter=100, random_state=42)
    # Обучаем модель
    model.fit(X_train, y_train)
    # Сохраняем предсказания в наш список по индексу
    predictions_results[index] = model.predict(X_test)

In [ ]:
# ========================================================
# ЗАДАНИЕ А: ПОСЛЕДОВАТЕЛЬНОЕ ОБУЧЕНИЕ
# ========================================================
print("\n--- Запуск ПОСЛЕДОВАТЕЛЬНОГО обучения ---")
start_seq = time.time() # Засекаем время

# Циклом перебираем от 0 до 8 и обучаем по очереди
for i in range(len(networks_settings)):
    layers = networks_settings[i][0]
    activation = networks_settings[i][1]

    # Вызываем функцию напрямую
    train_and_predict(i, layers, activation)

time_seq = time.time() - start_seq
print(f"Время последовательного обучения: {time_seq:.2f} сек.")

# Очищаем результаты перед многопоточным запуском, чтобы эксперимент был честным
for i in range(9):
    predictions_results[i] = None

In [ ]:
# ========================================================
# ЗАДАНИЕ Б: МНОГОПОТОЧНОЕ ОБУЧЕНИЕ
# ========================================================
print("\n--- Запуск МНОГОПОТОЧНОГО обучения ---")
start_thread = time.time()

ml_threads = []

# Создаем 9 потоков
for i in range(len(networks_settings)):
    layers = networks_settings[i][0]
    activation = networks_settings[i][1]

    # Создаем поток. Передаем функцию и её аргументы в виде кортежа (args)
    t = threading.Thread(target=train_and_predict, args=(i, layers, activation))
    ml_threads.append(t)
    t.start() # Запускаем поток параллельно

# Ждем завершения всех потоков (чтобы программа не пошла дальше, пока все не доучатся)
for t in ml_threads:
    t.join()

time_thread = time.time() - start_thread
print(f"Время многопоточного обучения: {time_thread:.2f} сек.")

In [ ]:
print("\n=== ИТОГИ СРАВНЕНИЯ ВРЕМЕНИ ===")
print(f"Последовательно: {time_seq:.2f} с.")
print(f"Потоки (Threading): {time_thread:.2f} с.")

In [ ]:
# ========================================================
# ЗАДАНИЕ В: ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ
# ========================================================
print("\n--- Построение графиков ---")
# Создаем сетку графиков 3x3
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle('Истинные значения vs Предсказания для 9 нейросетей', fontsize=16)

# Превращаем матрицу графиков 3x3 в одномерный список от 0 до 8
axes = axes.flatten()

for i in range(len(networks_settings)):
    ax = axes[i]
    layers = networks_settings[i][0]
    activation = networks_settings[i][1]

    # Достаем результаты предсказаний для конкретной сети
    predicted_values = predictions_results[i]

    if predicted_values is not None:
        # Рисуем точки (синие)
        ax.scatter(y_test, predicted_values, alpha=0.3, color='blue')
        # Строим идеальную линию y=x (красная пунктирная)
        ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)

        ax.set_title(f"Слои: {layers}, Активация: {activation}")
        ax.set_xlabel("Истинные значения")
        ax.set_ylabel("Предсказания")
        ax.grid(True)
    else:
        ax.set_title("Модель не обучена!")

plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.show()